In [ ]:
import re
import joblib
import pandas as pd

In [2]:
INPUT_FILE = "reviews_with_products.csv"
OUTPUT_FILE = "review_aspects.csv"
MODEL_FILE = "sentiment_model.joblib"

model = joblib.load(MODEL_FILE)
reviews = pd.read_csv(INPUT_FILE)

C:\Users\ikhta\AppData\Local\Temp\ipykernel_16804\3438072718.py:6: DtypeWarning: Columns (27: subtitle) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(INPUT_FILE)


In [3]:
# Add to these dictionaries after inspecting actual review language.
ASPECT_KEYWORDS = {
    "Quality and durability": [
        "quality", "durable", "durability", "sturdy", "flimsy",
        "cheaply made", "fell apart", "wear and tear", "well made",
        "poorly made", "broke easily"
    ],
    "Broken or missing parts": [
        "broken", "cracked", "missing piece", "missing pieces",
        "missing parts", "defective", "faulty", "does not work",
        "doesn't work", "out of the box"
    ],
    "Enjoyment and play": [
        "fun", "enjoyable", "enjoyed", "boring", "entertaining",
        "educational", "loves playing", "love playing", "great toy"
    ],
    "Ease of use and instructions": [
        "easy to use", "easy to assemble", "hard to assemble",
        "difficult to assemble", "instructions", "confusing",
        "complicated", "simple to use"
    ],
    "Delivery and arrival": [
        "delivery", "shipping", "arrived late", "arrived early",
        "arrived on time", "never arrived", "late arrival"
    ],
    "Packaging": [
        "packaging", "package", "damaged box", "box was damaged",
        "poorly packaged", "well packaged"
    ],
    "Value for money": [
        "worth the money", "value for money", "overpriced",
        "pricey", "good value", "not worth the price", "expensive"
    ],
    "Design and appearance": [
        "design", "colour", "color", "appearance", "looks great",
        "beautiful", "ugly", "attractive"
    ],
    "Safety": [
        "unsafe", "safety", "choking hazard", "sharp edges",
        "dangerous", "safe for children"
    ],
}


In [4]:
def split_sentences(text):
    # Simple baseline; abbreviations and punctuation can cause errors.
    return [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", str(text))
        if sentence.strip()
    ]

In [8]:
# One output row per review sentence and matching aspect.
aspect_records = []

for review_id, row in reviews.iterrows():
    review_text = row.get("review_text", "")

    if pd.isna(review_text):
        continue

    for sentence in split_sentences(review_text):
        sentence_lower = sentence.lower()

        for aspect, keywords in ASPECT_KEYWORDS.items():
            if any(keyword in sentence_lower for keyword in keywords):
                aspect_records.append({
                    "review_row_id": review_id,
                    "parent_asin": row.get("parent_asin"),
                    "product_title": row.get("product_title"),
                    "rating": row.get("rating"),
                    "overall_predicted_sentiment": row.get(
                        "predicted_sentiment"
                    ),
                    "review": row.get("review_text"),
                    "aspect": aspect,
                    "evidence_sentence": sentence,
                })



In [9]:
aspects = pd.DataFrame(aspect_records)

In [10]:
aspects.head(5)

,review_row_id,parent_asin,product_title,rating,overall_predicted_sentiment,review,aspect,evidence_sentence
0,0,B09QH7QJS7,"Schleich bayala, 5-Piece Starter Set with Fair...",5.0,positive,Granddaughters love them!. I purchased several...,Quality and durability,Schleich is well made & lasts forever.
1,1,B06XYKSKQP,Liberty Imports Electronic Walking Jurassic St...,3.0,neutral,"Arrived broken. It’s cute, but it arrived brok...",Broken or missing parts,Arrived broken.
2,1,B06XYKSKQP,Liberty Imports Electronic Walking Jurassic St...,3.0,neutral,"Arrived broken. It’s cute, but it arrived brok...",Broken or missing parts,"It’s cute, but it arrived broken & has some pr..."
3,1,B06XYKSKQP,Liberty Imports Electronic Walking Jurassic St...,3.0,neutral,"Arrived broken. It’s cute, but it arrived brok...",Broken or missing parts,As it arrived broken it’s definitely going back.
4,3,B007JWWUDW,Janod Story Train Circus,5.0,positive,Janod quality and very cute.... This was great...,Quality and durability,Janod quality and very cute....


In [ ]:
# read a specific review from the aspects dataframe in full
print(aspects.loc[4, 'review'])

Janod quality and very cute.... This was great for my daughters circus themed birthday! Good quality, super cute and really made a special touch for the cake table decorations. Plus now after the party she has this and the circus story box to play with! They are the great quality that you would expect from Janod  (wish they had more diversity in the people in the circus box however) this train set I would recommend!


In [17]:
if not aspects.empty:
    # Predict sentence-level sentiment in batches to limit memory use.
    predictions = []

    sentences = aspects["evidence_sentence"].tolist()
    batch_size = 512

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]
        predictions.extend(model.predict(batch))

    aspects["aspect_sentiment"] = predictions


In [20]:
aspects.to_csv(OUTPUT_FILE, index=False)


In [21]:
print("Reviews analysed:", len(reviews))
print("Aspect mentions detected:", len(aspects))
print("Saved to:", OUTPUT_FILE)

if not aspects.empty:
    print("\nMentions by aspect and sentiment:")
    print(pd.crosstab(aspects["aspect"], aspects["aspect_sentiment"]))

Reviews analysed: 147640
Aspect mentions detected: 140532
Saved to: review_aspects.csv

Mentions by aspect and sentiment:
aspect_sentiment              negative  neutral  positive
aspect                                                   
Broken or missing parts           8561     1729       289
Delivery and arrival               753      616       558
Design and appearance             5286     8075     11047
Ease of use and instructions      2525     2690      2533
Enjoyment and play                4310    10723     25796
Packaging                         2841     2326      1347
Quality and durability           12712    12635     13542
Safety                            1124      331       171
Value for money                   3359     3481      1172
